In [0]:
df = spark.read.table(
    "mythri_databricks.sales_silver.customers_silver"
)

display(df)

In [0]:
dbutils.widgets.text("source_table", "")

source_table = dbutils.widgets.get("source_table")

print("Source table:", source_table)

In [0]:
dbutils.widgets.text("source_table", "")
source_table = dbutils.widgets.get("source_table")
print("Source table:", source_table)

In [0]:
df = spark.read.table(source_table)
display(df)

In [0]:
spark.sql("SHOW TABLES IN mythri_databricks.customer_bronze").show(truncate=False)

In [0]:
spark.sql("SHOW TABLES IN mythri_databricks.sales_bronze").show(truncate=False)

In [0]:
df = spark.read.table("mythri_databricks.sales_bronze.customers_bronze")
display(df)

In [0]:
from pyspark.sql import functions as F

transformed_df = (
    df
    .dropDuplicates(["customer_id"])
    .withColumn(
        "purchase_category",
        F.when(F.col("purchase_amount") >= 7000, "High")
         .when(F.col("purchase_amount") >= 5000, "Medium")
         .otherwise("Low")
    )
)

display(transformed_df)

In [0]:
transformed_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("mythri_databricks.sales_bronze.customers_transformed")

In [0]:
spark.sql("""
SELECT *
FROM mythri_databricks.sales_bronze.customers_transformed
""").show()